In [2]:
import torchvision 
from torch import nn
from torch.utils.data import DataLoader
import torch
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms  # 新增这行

# ========== 修改这里：添加数据增强 ==========
# 训练集：带数据增强
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),     # 随机水平翻转
    transforms.RandomRotation(degrees=10),      # 随机旋转 ±10度
    transforms.RandomResizedCrop(size=32, scale=(0.8, 1.0)),  # 随机裁剪并缩放
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 随机调整亮度和对比度
    transforms.ToTensor(),                      # 转成张量
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],   # CIFAR10 的均值
                         std=[0.2470, 0.2435, 0.2616])    # CIFAR10 的标准差
])

# 测试集：只做标准化（不要数据增强）不然形成不了对比
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616])
])

# 准备数据集
train_data = torchvision.datasets.CIFAR10(
    root="data", 
    train=True, 
    transform=train_transform,  # 改这里
    download=True
)

test_data = torchvision.datasets.CIFAR10(
    root="data", 
    train=False, 
    transform=test_transform,   # 改这里
    download=True
)
# ==========================================

train_data_size = len(train_data)
test_data_size = len(test_data)
print(f"训练数据集的长度为：{train_data_size}")
print(f"测试数据集的长度为：{test_data_size}")

# DataLoader
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)

# 网络模型
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.model(x)

net = Net()

# 损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(net.parameters(), lr=learning_rate)

# 训练参数
total_train_step = 0
total_test_step = 0
epoch = 10

# TensorBoard
writer = SummaryWriter("logs_train")

for i in range(epoch):
    print(f"第{i+1}轮训练开始")
    
    # 训练步骤
    net.train()
    for data in train_dataloader:
        imgs, targets = data
        outputs = net(imgs)
        loss = loss_fn(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_train_step += 1
        if total_train_step % 100 == 0:
            print(f"训练次数：{total_train_step}, 损失值：{loss.item():.4f}")
            writer.add_scalar("train_loss", loss.item(), total_train_step)
    
    # 测试步骤
    net.eval()
    total_test_loss = 0
    total_accuracy = 0
    
    with torch.no_grad():
        for data in test_dataloader:
            imgs, targets = data
            outputs = net(imgs)
            
            loss = loss_fn(outputs, targets)
            total_test_loss += loss.item()
            
            predictions = outputs.argmax(dim=1)
            correct = (predictions == targets).sum().item()
            total_accuracy += correct
    
    avg_test_loss = total_test_loss / len(test_dataloader)
    test_accuracy = total_accuracy / test_data_size
    
    print(f"第{i+1}轮测试集 - Loss: {avg_test_loss:.4f}, 准确率: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    
    writer.add_scalar("test_loss", avg_test_loss, total_test_step)
    writer.add_scalar("test_accuracy", test_accuracy, total_test_step)
    total_test_step += 1
    
    torch.save(net.state_dict(), f"net_epoch_{i}.pth")
    print(f"模型已保存为 net_epoch_{i}.pth")

writer.close()
print("训练完成！")

Files already downloaded and verified
Files already downloaded and verified
训练数据集的长度为：50000
测试数据集的长度为：10000
第1轮训练开始
训练次数：100, 损失值：2.0569
训练次数：200, 损失值：2.0072
训练次数：300, 损失值：1.9175
训练次数：400, 损失值：1.9017
训练次数：500, 损失值：1.9931
训练次数：600, 损失值：1.8405
训练次数：700, 损失值：1.8971
第1轮测试集 - Loss: 1.8209, 准确率: 0.3507 (35.07%)
模型已保存为 net_epoch_0.pth
第2轮训练开始
训练次数：800, 损失值：1.6614
训练次数：900, 损失值：1.6004
训练次数：1000, 损失值：1.6186
训练次数：1100, 损失值：1.8048
训练次数：1200, 损失值：1.6163
训练次数：1300, 损失值：1.7130
训练次数：1400, 损失值：1.4830
训练次数：1500, 损失值：1.4093
第2轮测试集 - Loss: 1.5200, 准确率: 0.4428 (44.28%)
模型已保存为 net_epoch_1.pth
第3轮训练开始
训练次数：1600, 损失值：1.6100
训练次数：1700, 损失值：1.4324
训练次数：1800, 损失值：1.4284
训练次数：1900, 损失值：1.5082
训练次数：2000, 损失值：1.5263
训练次数：2100, 损失值：1.4931
训练次数：2200, 损失值：1.2933
训练次数：2300, 损失值：1.2450
第3轮测试集 - Loss: 1.5649, 准确率: 0.4554 (45.54%)
模型已保存为 net_epoch_2.pth
第4轮训练开始
训练次数：2400, 损失值：1.4405
训练次数：2500, 损失值：1.4939
训练次数：2600, 损失值：1.3674
训练次数：2700, 损失值：1.2520
训练次数：2800, 损失值：1.3692
训练次数：2900, 损失值：1.2619
训练次数：3000, 损失值：1.2999
训练次数：310

# 数据增强